In [3]:
# importing necessary libraries
import pandas as pd
import numpy as np
import TheMonsterScrap as ms # <-- real state scrapper framework

### **Basic extract methodology**

##### 1. **Determine target elements identifiers:**
After inspect the HTML structure of the target webpage, we can define the attributes of each desired HTML container for future use as follows:

In [13]:
# all listings are in a 'div' container with the attributes:
listings_container = 'div'
listings_attrs = {'class':'data'}

# price containers:
price_container = 'div'
price_container_attrs = {'class':'price'}

# location containers:
location_container = 'h2'
location_container_attrs = {'itemprop':'name', 'class':'title'}

# bedrooms containers:
bedrooms_container = 'span'
bedrooms_container_attrs = {'class':'rooms'}

# bathrooms containers:
bathrooms_container = 'span'
bathrooms_container_attrs = {'class':'bathrooms'}

# area containers:
area_container = 'span'
area_container_attrs = {'class':'areaBuilt'}

# url containers:
url_container = 'a' # no attrs

print('Attributes saved in variables.')

Attributes saved in variables.


##### 2. **Extract HTML text from the webpage and create a ScrapperMonster object with it:**

In [15]:
# html content
content = ms.HttpClient().get_html('https://www.icasas.mx/venta/habitacionales-departamentos-baja-california-sur-paz-2_3_4_0_19_0')
monster = ms.ScrapperMonster(content)
print('HTML content and Monster object ready to use.')

HTML content and Monster object ready to use.


##### 3. **Add the parsers of each target values to get:**

In [16]:
# add parsers to extract needed data
monster.add_parser('price', ms.PriceParser(price_container, attrs=price_container_attrs))
monster.add_parser('location', ms.LocationParser(location_container, attrs=location_container_attrs))
monster.add_parser('bedrooms', ms.BedroomsParser(bedrooms_container, attrs=bedrooms_container_attrs))
monster.add_parser('bathrooms', ms.BathroomsParser(bathrooms_container, attrs=bathrooms_container_attrs))
monster.add_parser('area', ms.AreaParser(area_container, attrs=area_container_attrs))
monster.add_parser('url', ms.UrlParser(url_container))
print('Parsers added to the Monster instance correctly.')

Parsers added to the Monster instance correctly.


##### 4. **Get all listings of the webpage and scrap desired information:**

In [17]:
# obtain all listings from the page
listings = monster.get_listings(listings_container, attrs=listings_attrs)
print('Listings getted.') 

# obtain the info of each listing
scraps = monster.unleash(listings)
print('Scrap done successfully.')

Listings getted.
Scrap done successfully.


In [18]:
# save scraps in a dataframe
df = ms.Exporter.to_dataframe(scraps)
print('Data saved in a Pandas df.')

Data saved in a Pandas df.


We can use a loop to automatize the process in the next pages:

In [19]:
for i in range(2,5):
    
    # get page content and create a monster with it
    html = ms.HttpClient().get_html(f'https://www.icasas.mx/venta/habitacionales-departamentos-baja-california-sur-paz-2_3_4_0_19_0/p_{i}')
    monsterr = ms.ScrapperMonster(html)
    
    # add parsers to the monster
    monsterr.add_parser('price', ms.PriceParser(price_container, attrs=price_container_attrs))
    monsterr.add_parser('location', ms.LocationParser(location_container, attrs=location_container_attrs))
    monsterr.add_parser('bedrooms', ms.BedroomsParser(bedrooms_container, attrs=bedrooms_container_attrs))
    monsterr.add_parser('bathrooms', ms.BathroomsParser(bathrooms_container, attrs=bathrooms_container_attrs))
    monsterr.add_parser('area', ms.AreaParser(area_container, attrs=area_container_attrs))
    monsterr.add_parser('url', ms.UrlParser(url_container))
    
    # get listings and scrap data from it
    depas_listings = monsterr.get_listings(listings_container, attrs=listings_attrs)
    depas_scrap = monsterr.unleash(depas_listings)
    
    # convert scrapped data into a dataframe and concatenate with the main df
    df2 = ms.Exporter.to_dataframe(depas_scrap)
    df = pd.concat([df, df2], axis=0)

In [20]:
# check the data was stored successfully into our main dataframe 
df = df.drop(['parkings'], axis=1)
df.head(3)

,price,location,bedrooms,bathrooms,area,url
0,"12,091,172.08 MX$","Departamento en Camino Del Pedregal, La Paz, ...",3,2,155m2,/propiedad/540e-872e-194f707-3842bff41d3-7909
1,"4,197,500 MX$destacado",Departamento en Calle Ignacio Altamirano & 16...,2,2,77m2,/propiedad/4c06-aeaa-195162f-10df2dbd3ad0-72a8
2,"4,197,500 MX$destacado",Departamento en Calle Ignacio Altamirano & 16...,2,2,77m2,/propiedad/ae77-b247-1951628-6a72839e57cb-7094


#### 5. **Export and process raw data:**

In [21]:
# export scrapped info to csv
df.to_csv('DEPARTMENTS_iCASAS.csv', index=False, encoding='utf-8')

### **Cleaning and formatting data workflow**

In [1]:
# export the cleaning and transforming framework
import TheAdaCleaner as ac

In [7]:
df2 = pd.read_csv('realstate_icasas.csv', encoding='utf-8').drop(['source'], axis=1)
df2.head(3)

,price,location,bedrooms,bathrooms,area,parkings,url,property_type
0,"13,961,053.5 MX$destacado","Casa en Residencial Puerta Azul, Chametla, Ba...",4.0,4.0,266.15m2,NaN,/propiedad/3b49-861b-194f6e0-64f1de6ee794-807b,Casa
1,"34,647,870 MX$","Casa en Camino Los Barriles - Boca Del Álamo,...",6.0,4.0,641m2,NaN,/propiedad/97ac-a8e5-1907fcf-2b196b11c09a-758e,Casa
2,"19,158,234 MX$","Casa en El Pescadero, La Paz",3.0,3.0,361.25m2,NaN,/propiedad/7101-ac2a-191392e-9f61a6489b3-7f78,Casa


In [8]:
# create and Ada object with the columns we need to clean:
#       ADMITTED COLUMNS
#       - 'price'
#       - 'location'
#       - 'area'
#       - 'parkings' <-- especially useful for data from website 1
ada = ac.Ada(columns_to_clean=['price', 'location', 'area'])

# execute Ada's Clean method to clean and reformating data
df2 = ada.clean(df2)
df2.head() 

,price,location,bedrooms,bathrooms,area,parkings,url,property_type,listed_in,full_location,register_date
0,13961053.5,"Residencial Puerta Azul, La Paz, Baja Californ...",4.0,4.0,266.15,NaN,/propiedad/3b49-861b-194f6e0-64f1de6ee794-807b,Casa,MXN,"Casa en Residencial Puerta Azul, Chametla, Ba...",2025-04-02 13:44
1,34647870.0,"lamo, La Paz, Baja California Sur, Mexico",6.0,4.0,641.00,NaN,/propiedad/97ac-a8e5-1907fcf-2b196b11c09a-758e,Casa,MXN,"Casa en Camino Los Barriles - Boca Del Álamo,...",2025-04-02 13:44
2,19158234.0,"El Pescadero, La Paz, Baja California Sur, Mexico",3.0,3.0,361.25,NaN,/propiedad/7101-ac2a-191392e-9f61a6489b3-7f78,Casa,MXN,"Casa en El Pescadero, La Paz",2025-04-02 13:44
3,6766525.2,"Calle La Trinidad, La Paz, Baja California Sur...",2.0,1.0,174.00,NaN,/propiedad/fc5c-8d66-1922b23-d69cf95a5681-791f,Casa,MXN,"Casa en Calle La Trinidad, Brisas Del Pacífic...",2025-04-02 13:44
4,11800656.9,"Calle Colina Del Sol, La Paz, Baja California ...",2.0,2.0,230.00,NaN,/propiedad/e01a-9b22-19105ac-2a9893803c04-7723,Casa,MXN,"Casa en Calle Colina Del Sol, Colina Del Sol,...",2025-04-02 13:44


Now, dataset is reformatted and ready to use, either for loading into a database or training ML models.